In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

In [ ]:
allowBending = False

name = 'circles'
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  

pressure = 2


# for dim in np.linspace(0.2, 3.97, 30):
def trunc(values, decs=0):
    return np.trunc(values*10**decs)/(10**decs)
dim = trunc(np.linspace(0.2, 3.97, 30)[15], 2)
with open('../data/Circles/CIR_Diam_{}.json'.format(dim), 'r') as f:
    data = json.load(f)
fusedVertices = data['FusedVertices']
fusedVertices = [True if vx == 1 else False for vx in fusedVertices]
V = data['Vertices']
F = data['Faces']
m = MeshFEM.Mesh(V, F)
for vx in m.boundaryVertices():
    fusedVertices[vx] = False
ipu = inflation.InflatablePeriodicUnit(m, fusedVertices)

finalMarkers = np.where(np.array(fusedVertices) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = pressure

benchmark.reset()

opts.niter = 500
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
benchmark.report()

az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx)
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
variable = dim
if not allowBending:
    stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 100, az_optimizer, hessianShift = 1e-8, fixedVars = [], filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable))
    np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
    np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)

points = visualize_average_deformation_gradient(ipu, 100, filename = "{}/average_deformation_gradient_{}_{}.png".format(result_folder, name, variable))

render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/render_{}_{}.png".format(result_folder, name, variable))

np.save("{}/scale_factors_{}_{}.npy".format(result_folder, name, variable), get_deformation_scale_factors(ipu))
np.save("{}/kappa_{}_{}.npy".format(result_folder, name, variable), az_ipu.getVars()[-2])

In [ ]:
min(stiffness_values)

In [ ]:
max(stiffness_values)

In [ ]:
cr.success

In [ ]:
viewer = TriMeshViewer(az_ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

